<a href="https://colab.research.google.com/github/CompOmics/lsabd-machine-learning-tutorials/blob/main/notebooks/4-dimensionality-reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dimensionality reduction and latent spaces

Biomedical datasets often contain thousands of features, such as all proteins measured in a sample. Dimensionality reduction methods summarize such data in two dimensions, so that we can look at it in a scatter plot. In this notebook, we apply three common methods to a real proteomics dataset, and then use the same idea to look inside a foundation model. We will cover:

1. Loading and preparing a proteomics dataset
2. Comparing PCA, t-SNE, and UMAP
3. Local and global structure: the effect of the hyperparameters
4. Looking inside a foundation model with UMAP

Note that throughout the notebook, there are links to the relevant documentation for the functions and classes used. Click on the links and explore the documentation to deepen your understanding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set the style for our visualizations
sns.set_style("whitegrid")
sns.set_palette("deep")

## 1. Loading and preparing a proteomics dataset

The data come from a [study on feminizing gender-affirming hormone therapy (GAHT)](https://www.nature.com/articles/s41591-025-04023-9). The blood plasma of transgender individuals was measured before and after six months of treatment with estradiol plus one of two anti-androgens: cyproterone acetate (CPA) or spironolactone (SPIRO). The dataset also contains samples from pregnant women (first and third trimester) and a few cisgender controls. About 5,400 proteins were measured per sample with the Olink proximity extension assay. The data are available in the [PRIDE database](https://www.ebi.ac.uk/pride/archive/projects/PAD000016).

The data come in two files: a table with the protein measurements, and a metadata table that describes each sample. Let's load both with the pandas [`read_csv()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) function, as in notebook 1:

In [ ]:
data = pd.read_csv("https://github.com/CompOmics/lsabd-machine-learning-tutorials/raw/refs/heads/main/data/PAD000016_data.csv")
metadata = pd.read_csv("https://github.com/CompOmics/lsabd-machine-learning-tutorials/raw/refs/heads/main/data/GAHT_pregnancy_metadata_SDRF.tsv", sep="\t")

data.head()

The measurement table is in **long format**: every row contains one measurement of one protein (`Assay`) in one sample (`SampleID`). The protein levels are in the `PCNormalizedNPX` column. The table also contains measurements of quality control samples and control assays, which are not real samples or proteins. We remove them first:

In [ ]:
# Keep only real samples, and remove the control assays
data = data[data["SampleType"] == "SAMPLE"]
data = data[~data["Assay"].str.contains("control")]

For dimensionality reduction, we need the data in **wide format**: one row per sample and one column per protein. The pandas [`pivot_table()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.pivot_table.html) method does this reshaping; it is the opposite of the `melt()` method that we used for the box plots in notebook 1. Some proteins were measured more than once in the same sample; `aggfunc="mean"` takes the average of these measurements.

In [ ]:
# Reshape the data to wide format: one row per sample, one column per protein
features = data.pivot_table(
    index="SampleID", columns="Assay", values="PCNormalizedNPX", aggfunc="mean"
)
features.shape

Finally, we look up the group (phenotype) of each sample in the metadata, in the same order as the rows of `features`, and check for missing values with the `isnull()` method from notebook 1:

In [ ]:
labels = metadata.set_index("source name")["factor value[phenotype]"].reindex(features.index)

print(f"Missing protein values: {features.isnull().sum().sum()}")
print(f"Samples without a group: {labels.isnull().sum()}")
labels.value_counts()

*Question: How many samples and features does the dataset contain? Why can we not simply look at this data in a scatter plot, as we did with the features in notebooks 1 and 2?*

Write your answer here:

...

*Question: Do we need the group labels to perform dimensionality reduction?*

Write your answer here:

...

*Question: In notebooks 1 and 3, we scaled the features before training a model. Here, we do not. Is scaling (or normalization) necessary for dimensionality reduction?*

Write your answer here:

...

## 2. Comparing PCA, t-SNE, and UMAP

We now reduce the 5,416 protein features to two dimensions with three methods:

- [`PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) (principal component analysis) from scikit-learn,
- [`TSNE`](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) (t-distributed stochastic neighbor embedding) from scikit-learn, and
- [`UMAP`](https://umap-learn.readthedocs.io/en/latest/) (uniform manifold approximation and projection) from the `umap-learn` package.

All three follow the same pattern as the scalers in notebook 1: you create the model with its settings, and then call its `fit_transform()` method on the data. As these methods are unsupervised and we do not evaluate a prediction, there is no need for a train/test split here.

*Task: Reduce the features to two dimensions with each of the three methods, using `fit_transform(features)`. Use `PCA(n_components=2)`, `TSNE(n_components=2, perplexity=30, random_state=42)`, and `umap.UMAP(n_components=2, random_state=42)`. Store the results in the variables `pca_result`, `tsne_result`, and `umap_result`.*

*UMAP may print a warning about `n_jobs` being overridden: you can safely ignore it.*

In [ ]:
# Reduce the features to two dimensions with PCA, t-SNE, and UMAP
# Store the results in pca_result, tsne_result, and umap_result

import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE


Let's plot the three results side by side, with the points colored by group:

In [ ]:
fig, axes = plt.subplots(figsize=(20, 6), ncols=3)
for ax, result, title in zip(axes, [pca_result, tsne_result, umap_result], ["PCA", "t-SNE", "UMAP"]):
    sns.scatterplot(x=result[:, 0], y=result[:, 1], hue=labels, ax=ax, legend=(title == "UMAP"))
    ax.set_title(title)
axes[2].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

*Question: Which groups of samples are clearly separated, and which overlap? Does that make sense, given the study design?*

Write your answer here:

...

*Question: How do the three plots differ? Are these differences expected, given what you learned about PCA, t-SNE, and UMAP?*

Write your answer here:

...

## 3. Local and global structure

Dimensionality reduction methods have to make choices about what to preserve. **Local structure** refers to the relationships between nearby points: which samples are each other's nearest neighbors. **Global structure** refers to the overall layout: how far apart groups of samples are, and how they are positioned relative to each other. PCA, a linear method, focuses on global structure. t-SNE focuses on local structure, and UMAP tries to balance both.

The balance between local and global structure is controlled by a hyperparameter: the **perplexity** for t-SNE, and the **number of neighbors** (`n_neighbors`) for UMAP. Both roughly set how many neighbors each point "looks at" when the method builds its map. Let's see what happens when we change the perplexity of t-SNE:

In [ ]:
perplexity_values = [5, 10, 20, 30, 50, 100]

fig, axes = plt.subplots(figsize=(20, 12), nrows=2, ncols=3)
for ax, perplexity in zip(axes.flatten(), perplexity_values):
    result = TSNE(n_components=2, perplexity=perplexity, random_state=42).fit_transform(features)
    sns.scatterplot(x=result[:, 0], y=result[:, 1], hue=labels, ax=ax, legend=False)
    ax.set_title(f"t-SNE, perplexity = {perplexity}")
plt.tight_layout()
plt.show()

*Question: How does the perplexity affect the t-SNE plot?*

Write your answer here:

...

*Task: Repeat the analysis for UMAP. Copy the code of the previous cell, and change it so that it uses `umap.UMAP(n_components=2, n_neighbors=..., random_state=42)` instead of t-SNE. Try the following values for `n_neighbors`: `[2, 5, 15, 30, 50, 100]`. Do not forget to change the variable names and the plot titles.*

In [ ]:
# Repeat the analysis for UMAP with different values for n_neighbors: [2, 5, 15, 30, 50, 100]


*Question: How does the number of neighbors affect the UMAP plot? How would you choose a good value for the perplexity or the number of neighbors?*

Write your answer here:

...

*Question: In a t-SNE or UMAP plot, can you conclude that a large cluster contains more diverse samples than a small cluster? Or that two clusters that are far apart are more different than two clusters that are close together?*

Write your answer here:

...

## 4. Looking inside a foundation model with UMAP

So far, we applied dimensionality reduction to a **feature space**: the measured protein levels, where every column has a name and a meaning. A neural network also builds internal representations of the data: for every sample, the output of a hidden layer is again a row of numbers. Such a **latent space** is just as high-dimensional, but its dimensions are learned by the model and have no name or meaning that we can read.

Dimensionality reduction methods work on any high-dimensional table, so we can also apply UMAP to a model's latent space. Samples that the model treats alike then end up close together. This is a common way to look inside a trained model.

**OmicsFM** is a foundation model for proteomics and transcriptomics data, developed by the CompOmics group ([Heyndrickx et al., bioRxiv 2026](https://doi.org/10.64898/2026.08.25.747021)). As explained in the lectures, it was pretrained on large amounts of public data **without labels**: the model learned to predict masked protein abundances from the rest of a sample, much like a language model learns to predict missing words. Along the way, it learned a representation for every sample and for every protein. The [OmicsFM web viewer](https://compomics.github.io/OmicsFM_webview/) shows UMAP plots of these latent spaces.

*Task: Open the [OmicsFM web viewer](https://compomics.github.io/OmicsFM_webview/) and go to **Proteomics → Samples**. Each point is one proteomics sample from the public PRIDE database, positioned by a UMAP of the model's sample representation. Hover over points to see their metadata, click a point to see its ten nearest neighbors, click a class in the legend to show only that class, and use "Show projects" to color the samples by the project they come from. Then, answer the questions below.*

*Question: The points are colored by tissue. Remember that the model never saw these tissue labels during training. Do samples from the same tissue end up close together? What does that tell you about what the model has learned?*

Write your answer here:

...

*Question: Click a sample and look at its ten nearest neighbors. Are they from the same tissue, from the same project, or a mix? Note that these neighbors are computed in the original, high-dimensional latent space, not in the 2D plot.*

Write your answer here:

...

*Question: Two tissue clusters lie far apart in the plot. Can you conclude that these tissues have very different proteomes?*

Write your answer here:

...

*Task: Now go to **Proteomics → Proteins**. Here, each point is not a sample, but a protein: proteins that behave similarly across many samples end up close together. Hover over a few points to see which proteins they are.*

*Question: On the Samples page, each point was a sample; here, each point is a protein. How could you make such a map of proteins with the data from section 1?*

Write your answer here:

...

## Summary

- Dimensionality reduction summarizes high-dimensional data in two dimensions, to explore it visually. It is unsupervised: labels are only used afterwards, to interpret the plot.
- PCA is linear and preserves global structure. t-SNE focuses on local structure, and UMAP balances local and global structure.
- The perplexity (t-SNE) and the number of neighbors (UMAP) change the balance between local and global structure. Cluster sizes and distances between clusters in t-SNE and UMAP plots should be interpreted with care.
- The same methods can visualize the latent space of a model, such as a foundation model. Samples or proteins that the model treats alike end up close together, but closeness can reflect biology as well as technical batch effects.